# Atlas Python Static Analysis Tutorial

Atlas builds a complete navigable tree of your Python codebase through AST-based reconnaissance.

## 1. Basic Setup

Build a project tree by pointing Atlas at your target directory:

In [1]:
from analyzer import build_complete_atlas

# Build complete project tree
project = build_complete_atlas('sample_files')
print(f"Project: {project.name}")

Project: sample_files


## 2. Visualize Project Structure

Display the complete project tree:

In [2]:
# Print full tree structure
project.print()

Project(sample_files)
  Package(api)
    Package(endpoints)
      Module(product_endpoints)
        Class(ProductEndpoints)
          Function(__init__)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
          Function(create_category)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(name)
            Argument(description)
              MissingArgumentTypeHint(ArgumentNode)
          Function(get_category)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(category_id)
              MissingArgumentTypeHint(ArgumentNode)
          Function(create_product)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(name)
            Argument(price)
            Argument(category_id)
            Argument(description)
              MissingArgumentTypeHint(ArgumentNode)
          Function(get_product)
            Argument(

## 3. Navigation API

Navigate through packages, modules, classes, and functions:

In [3]:
# List all packages
packages = project.list_packages()
print(f"Found {len(packages)} packages")

# Get specific package
models_pkg = project.get_package('models')
print(f"Package: {models_pkg.fqn}")

# List modules in package
modules = models_pkg.list_modules()
print(f"Modules: {[m.name for m in modules]}")

Found 5 packages
Package: sample_files.models
Modules: ['order', 'product', 'user']


## 4. Class Discovery

Explore classes and their methods:

In [4]:
# Get a module and list its classes
user_module = models_pkg.get_module('user')
classes = user_module.list_classes()
print(f"Classes in {user_module.name}: {[c.name for c in classes]}")

# Get specific class
user_class = user_module.get_class('User')
print(f"\nClass: {user_class.fqn}")

# List methods
methods = user_class.list_methods()
print(f"Methods: {[m.name for m in methods]}")

Classes in user: ['User', 'UserProfile']

Class: sample_files.models.user.User
Methods: ['__init__', 'get_email', 'set_email', 'add_role', 'has_role', 'get_roles', 'activate', 'deactivate']


## 5. Function Signatures

Analyze function arguments and return types:

In [5]:
# Get a method and examine its signature
init_method = user_class.get_method('__init__')
print(f"Method: {init_method.name}")

# List arguments
args = init_method.list_arguments()
print(f"\nArguments ({len(args)}):")
for arg in args:
    type_info = arg._type if hasattr(arg, '_type') and arg._type else None
    type_str = type_info.name if type_info else "No type hint"
    print(f"  {arg.name}: {type_str}")

# Check return type
returns = init_method.list_returns()
if returns:
    ret = returns[0]
    type_info = ret._type if hasattr(ret, '_type') and ret._type else None
    print(f"\nReturn type: {type_info.name if type_info else 'None'}")

Method: __init__

Arguments (5):
  self: No type hint
  user_id: str
  email: str
  username: No type hint
  password: No type hint


## 6. Attribute Discovery

Find class and instance attributes:

In [6]:
# List class attributes
class_attrs = user_class.list_class_attributes()
print(f"Class attributes ({len(class_attrs)}):")
for attr in class_attrs:
    print(f"  {attr.name}")

# List instance attributes
instance_attrs = user_class.list_instance_attributes()
print(f"\nInstance attributes ({len(instance_attrs)}):")
for attr in instance_attrs:
    type_info = attr._type if hasattr(attr, '_type') and attr._type else None
    type_str = type_info.name if type_info else "No type hint"
    print(f"  {attr.name}: {type_str}")

Class attributes (0):

Instance attributes (5):
  email: No type hint
  username: No type hint
  password: No type hint
  is_active: No type hint
  roles: No type hint


## 7. Import Analysis

Discover module imports:

In [7]:
# List all imports in a module
imports = user_module.list_imports()
print(f"Found {len(imports)} import statements")

# Examine individual imports
for imp in imports[:3]:  # Show first 3
    aliases = imp.list_aliases()
    for alias in aliases:
        # Access AST data for import details
        original = alias.source_data.name
        local = alias.source_data.asname if alias.source_data.asname else original
        if local != original:
            print(f"  {local} (alias for {original})")
        else:
            print(f"  {original}")

Found 3 import statements
  Optional
  List
  datetime
  BaseEntity


## 8. Violation Detection

Find type hint violations and code issues:

In [8]:
# Check for violations on arguments
violations = []
for arg in args:
    if hasattr(arg, '_violations') and arg._violations:
        violations.extend(arg._violations)

print(f"Found {len(violations)} violations in {init_method.name}")
for v in violations[:3]:  # Show first 3
    print(f"  {v.__class__.__name__}")

Found 3 violations in __init__
  MissingArgumentTypeHint
  MissingArgumentTypeHint
  MissingArgumentTypeHint


## 9. Recursive Discovery

Use `list_all_*()` methods to traverse the entire tree:

In [9]:
# Find all classes in the entire project
all_classes = project.list_all_classes()
print(f"Total classes in project: {len(all_classes)}")

# Find all methods across all classes
all_methods = project.list_all_methods()
print(f"Total methods in project: {len(all_methods)}")

# Find all functions (module-level)
all_functions = project.list_all_functions()
print(f"Total functions in project: {len(all_functions)}")

Total classes in project: 27
Total methods in project: 142
Total functions in project: 15


## 10. JSON Serialization

Export project analysis to JSON:

In [10]:
# Export to JSON file
project.save_dump('project_analysis.json')

# Or get as dict for programmatic use
data = project.dump()
print(f"Project type: {data['type']}")
print(f"Project name: {data['name']}")
print(f"Number of packages: {len(data.get('children', []))}")

✓ Project tree serialized to: project_analysis.json
  File size: 78,106 bytes
  Nodes serialized: 246
Project type: Project
Project name: sample_files
Number of packages: 5


## 11. Practical: Find All Type Violations

Scan entire project for missing type hints:

In [11]:
# Collect all violations across the project
all_violations = []

# Check all arguments
for arg in project.list_all_arguments():
    if hasattr(arg, '_violations') and arg._violations:
        for v in arg._violations:
            all_violations.append({
                'type': v.__class__.__name__,
                'location': arg.fqn,
                'entity': 'argument'
            })

# Check all returns
for ret in project.list_all_returns():
    if hasattr(ret, '_violations') and ret._violations:
        for v in ret._violations:
            all_violations.append({
                'type': v.__class__.__name__,
                'location': ret.fqn,
                'entity': 'return'
            })

print(f"Total violations: {len(all_violations)}")
print(f"\nBreakdown:")
from collections import Counter
violation_counts = Counter(v['type'] for v in all_violations)
for vtype, count in violation_counts.most_common():
    print(f"  {vtype}: {count}")

Total violations: 236

Breakdown:
  MissingArgumentTypeHint: 236


## 12. Practical: Analyze Method Complexity

Find methods with many arguments (potential complexity issues):

In [12]:
# Analyze all methods for argument count
complex_methods = []

for method in project.list_all_methods():
    args = method.list_arguments()
    # Filter out 'self' and 'cls'
    param_count = len([a for a in args if a.name not in ('self', 'cls')])
    
    if param_count >= 4:  # Threshold for "complex"
        complex_methods.append({
            'fqn': method.fqn,
            'params': param_count
        })

# Sort by parameter count
complex_methods.sort(key=lambda x: x['params'], reverse=True)

print(f"Found {len(complex_methods)} methods with 4+ parameters\n")
for m in complex_methods[:5]:  # Show top 5
    print(f"  {m['fqn']}: {m['params']} params")

Found 10 methods with 4+ parameters

  sample_files.models.product.Product.__init__: 5 params
  sample_files.services.email_service.EmailService.send_email: 5 params
  sample_files.services.payment_service.PaymentProcessor.process_payment: 5 params
  sample_files.api.endpoints.product_endpoints.ProductEndpoints.create_product: 4 params
  sample_files.models.user.User.__init__: 4 params
